# SparkSession + RDD Basics
**Topic:** Introduction to Apache Spark Core Concepts  
**Date:** March 30, 2026  
**GitHub:** de-upskilling/01-PySpark/basics/02_sparksession_rdd.ipynb

## 1. What is Apache Spark and Why is it Faster than Hadoop?

Apache Spark is an open-source, distributed computing framework 
designed for fast, large-scale data processing.

**Why faster than Hadoop?**
- Hadoop reads and writes to **disk** after every single step → very slow
- Spark processes data **in-memory (RAM)** → up to 100x faster
- Spark uses a **DAG (Directed Acyclic Graph)** execution engine that 
  optimizes the full execution plan before running anything
- Hadoop follows a rigid **Map → Reduce** two-step model
- Spark can chain multiple operations without writing intermediate results to disk

**Real DE use case:**  
When processing 500 million rows of sales data, Hadoop would write 
intermediate results to disk after every transformation. Spark keeps 
everything in memory and processes it in one optimized flow — 
finishing in minutes instead of hours.

In [1]:
# Verify PySpark is installed and check version
import pyspark
print("PySpark Version:", pyspark.__version__)

PySpark Version: 3.4.1


## 2. SparkSession — Entry Point to PySpark

SparkSession is the **single entry point** to all PySpark functionality.  
Think of it as the **"gateway"** — before you can do anything in Spark, 
you must create a SparkSession first.

**Key points:**
- Introduced in Spark 2.0 — replaced the older SparkContext
- One SparkSession per application (use `getOrCreate()` to avoid duplicates)
- `builder` → used to configure and create the session
- `appName` → name of your Spark application (shows in Spark UI)
- `master("local[*]")` → run locally using ALL available CPU cores on your laptop
- `getOrCreate()` → creates new session or returns existing one if already created

**In production (Azure Databricks):**  
SparkSession is already created for you automatically — 
you just use `spark` directly without creating it manually.

In [4]:
from pyspark.sql import SparkSession

#Create SparkSession
spark = SparkSession.builder.appName('SparkSession_RDD_Basics')\
    .master("local[*]").getOrCreate()

print("SparkSession created successfully!")
print("App Name:", spark.sparkContext.appName)
print("Spark Version:", spark.version)
print("Master:", spark.sparkContext.master)

SparkSession created successfully!
App Name: SparkSession_RDD_Basics
Spark Version: 3.4.1
Master: local[*]


## 3. SparkContext vs SparkSession

| Feature | SparkContext | SparkSession |
|---------|-------------|--------------|
| Introduced in | Spark 1.x | Spark 2.x |
| Purpose | Low-level entry point for RDD operations | Unified entry point for RDD + DataFrame + SQL |
| Usage today | Mostly replaced | ✅ Use this always |
| How to access | Created directly | Access via `spark.sparkContext` |
| Supports SQL? | ❌ No | ✅ Yes |
| Supports DataFrames? | ❌ No | ✅ Yes |

**Key takeaway:**  
Always create SparkSession. SparkContext is automatically 
created inside SparkSession and can be accessed via `spark.sparkContext`.

In [5]:
# Access SparkContext from SparkSession
sc = spark.sparkContext

print("SparkContext:", sc)
print("App Name:", sc.appName)
print("Master:", sc.master)
print("Number of CPU cores available:", sc.defaultParallelism)

SparkContext: <SparkContext master=local[*] appName=SparkSession_RDD_Basics>
App Name: SparkSession_RDD_Basics
Master: local[*]
Number of CPU cores available: 16


## 4. RDD — Resilient Distributed Dataset

RDD is the **fundamental, low-level data structure** of Apache Spark.

**Breaking down the name:**
- **Resilient** → Fault-tolerant. If a node fails, Spark can 
  recreate the lost data using RDD lineage (it remembers every 
  transformation applied)
- **Distributed** → Data is split into partitions and spread 
  across multiple nodes/cores for parallel processing
- **Dataset** → A collection of data (like a list or table)

**Key characteristics:**
- **Immutable** → Once created, cannot be changed. 
  Every transformation creates a NEW RDD
- **Lazy evaluation** → Transformations are not executed immediately. 
  Spark waits until an Action is called
- **In-memory** → Processed in RAM for speed

**Two ways to create an RDD:**
1. `sc.parallelize()` → from a Python list
2. `sc.textFile()` → from a file

**Important:** RDDs are mostly replaced by DataFrames in modern Spark 
because DataFrames are faster (Catalyst Optimizer) and easier to use. 
But understanding RDDs is important for interviews!

## Why Do We Still Talk About RDDs?

You might ask: *"If DataFrames are faster and easier, why learn RDDs?"*

**3 reasons RDDs still matter:**

1. **Under the Hood** → DataFrames are actually built *on top* of RDDs. 
   When you run a DataFrame query, Spark compiles it down to RDD code internally.

2. **Unstructured Data** → RDDs are powerful when your data has no schema 
   (e.g., raw text files, media, complex logs) and you need low-level control.

3. **Debugging** → Understanding RDD partitions and lineage helps you debug 
   performance issues like data skew and excessive shuffling in DataFrames.

---

**Simply explained:**

- **Coding with RDDs tells Spark "HOW" to do something** →  
  You explicitly define each transformation step with imperative code, 
  giving you full control over the execution logic.

- **Coding with DataFrames tells Spark "WHAT" to do** →  
  You declare your intent using high-level SQL-like operations, 
  and the Catalyst Optimizer figures out the best "how" automatically.

**Example:**  
With RDDs → you manually say *"filter these tuples, then map this function, 
then reduce with this logic"*  
With DataFrames → you simply say *"select these columns where this condition 
is true"* and Spark optimizes the execution plan for you.

In [7]:
#___________ WAY 1: Create RDD from a Python list __________________

numbers = [1,3,1,0,2,8,1,0,9,7]
rdd = sc.parallelize(numbers)

print("RDD created:", rdd)
print("Number of partitions:", rdd.getNumPartitions())
print("Type:", type(rdd))

RDD created: ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:287
Number of partitions: 16
Type: <class 'pyspark.rdd.RDD'>


## 5. Basic RDD Operations

RDD operations are of two types:

**Transformations** (Lazy — not executed immediately):
- `map()` → apply a function to every element
- `filter()` → keep only elements matching a condition
- `flatMap()` → like map but flattens the result

**Actions** (Trigger execution):
- `collect()` → bring ALL data to driver — ⚠️ use carefully on big data!
- `count()` → total number of elements
- `first()` → first element
- `take(n)` → first n elements
- `sum()` → sum of all elements

**Rule of thumb:**  
Transformations are lazy (just build the plan).  
Actions actually execute the plan and return results.

In [9]:
# _____________ Basic RDD Actions ___________

print("All elements:", rdd.collect())
print("Count:",rdd.count())
print("First element:", rdd.first())
print("First 3 elements:",rdd.take(3))
print("Sum:",rdd.sum())


#_______________ RDD Transformations _____________
# map() -> square every number

squared = rdd.map(lambda x:x **2)
print("\nSquared:", squared.collect())

#filter() -> keep only even numbers
evens = rdd.filter(lambda x:x%2 ==0)
print("Even Numbers:",evens.collect())

#Chaining -> filter evens then multiply by 10
result = rdd.filter(lambda x:x%2 ==0).map(lambda x:x*10)
print("Evens x 10:", result.collect())

All elements: [1, 3, 1, 0, 2, 8, 1, 0, 9, 7]
Count: 10
First element: 1
First 3 elements: [1, 3, 1]
Sum: 32

Squared: [1, 9, 1, 0, 4, 64, 1, 0, 81, 49]
Even Numbers: [0, 2, 8, 0]
Evens x 10: [0, 20, 80, 0]


## 6. Creating RDD from a Text File

In real Data Engineering, we read data from files — not Python lists.  
`sc.textFile()` reads a file and creates an RDD where 
**each line becomes one element**.

**DE Use case:**  
Reading raw log files, config files, or any unstructured text data 
before converting to structured DataFrames.

In [11]:
#First create a sample text file
with open("pipeline_log.txt", "w") as f:
    f.write("Pipeline started at 09:00 AM\n")
    f.write("Reading source: sales_data.csv\n")
    f.write("Rows processed: 50000\n")
    f.write("Pipeline completed successfully\n")


# Read file into RDD
text_rdd = sc.textFile("pipeline_log.txt")

print("Total lines:", text_rdd.count())
print("All lines:")
for line in text_rdd.collect():
    print("->",line)

Total lines: 4
All lines:
-> Pipeline started at 09:00 AM
-> Reading source: sales_data.csv
-> Rows processed: 50000
-> Pipeline completed successfully


## 7. Why DataFrames Replaced RDDs

| Feature | RDD | DataFrame |
|---------|-----|-----------|
| Schema | ❌ No column names | ✅ Named columns + data types |
| Optimization | ❌ Manual | ✅ Catalyst Optimizer (automatic) |
| SQL support | ❌ No | ✅ Yes — spark.sql() |
| Ease of use | ❌ Need map/filter manually | ✅ Easy API like pandas |
| Performance | Slower | Faster |
| When to use | Legacy code, low-level ops | ✅ Always in modern Spark |

## 🔑 Key Takeaways
- Always create **SparkSession** first — it's your entry point
- `local[*]` = use all CPU cores on your laptop
- RDD = low-level, immutable, distributed collection
- **Transformations** are lazy, **Actions** trigger execution
- RDDs are important to understand conceptually for interviews
- In real DE work → always use **DataFrames** over RDDs

## 💼 Interview Question
**Q: Why is Spark faster than Hadoop?**  
A: Spark processes data in-memory (RAM) whereas Hadoop writes 
intermediate results to disk after every MapReduce step. 
This makes Spark up to 100x faster for iterative workloads.